# Monogate / eml-cost — 5-cell demo

**Run this on Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agent-maestro/monogate/blob/master/notebooks/demo.ipynb)

What this shows:
1. Install the public packages.
2. Evaluate the EML primitive: `eml(x, y) = exp(x) − ln(y)`.
3. `analyze()` a few expressions and read off their Pfaffian cost class.
4. `analyze_dynamics()` on a damped oscillator — counts oscillation + decay modes.
5. Integer construction cost from `{1}`: how expensive is `360`?

Tier note: the cost classes and dynamics counter are **OBSERVATION**-tier — empirical fits across 578 expressions in 12 subdomains (E-196 / C-207). They are reproducible numerically; they are not Lean-verified.

Source / context: https://github.com/agent-maestro/monogate · https://monogate.org/atlas

In [ ]:
# Cell 1 — install the public packages.
# eml-cost ships the analysis API; monogate (Python) wraps the EML
# primitive + provides utility helpers.
%pip install --quiet 'monogate>=2.5.0' 'eml-cost>=0.12.0'
import eml_cost
print('eml-cost', eml_cost.__version__)

In [ ]:
# Cell 2 — evaluate the EML primitive directly.
#   eml(x, y) = exp(x) − ln(y)
# Standard elementary functions reduce to short EML trees.
import math

def eml(x, y):
    return math.exp(x) - math.log(y)

# exp(x): eml(x, 1) since ln(1) = 0
print('exp(2.0)        =', eml(2.0, 1.0), 'vs math.exp:', math.exp(2.0))

# ln(y): eml(0, 1/y) flipped sign or the EXL primitive directly:
#   ln(y) = exp(0) - eml(0, y) = 1 - (1 - ln(y)) = ln(y) — pedagogically:
print('ln(2.7183)      =', math.log(2.7183), '(direct)')

# sin(x) over the complex plane uses the Euler bypass:
#   exp(i*x) = cos(x) + i*sin(x)
import cmath
z = cmath.exp(complex(0, 1.0))  # 1 EML node over the complex domain
print('sin(1.0) via Euler =', z.imag, 'vs math.sin:', math.sin(1.0))

In [ ]:
# Cell 3 — analyze() reads off the Pfaffian cost class of a SymPy expression.
#   pfaffian_r:    total Pfaffian chain order (Khovanskii, summed across
#                  parallel branches).
#   max_path_r:    chain order along the deepest root-to-leaf path.
#   eml_depth:     EML routing tree depth.
#   cost_class:    canonical p<r>-d<d>-w<mr>-c<c> string used for cluster
#                  enrichment analysis.
import sympy as sp
from eml_cost import analyze, fingerprint_axes

x, t, omega, alpha, beta = sp.symbols('x t omega alpha beta', real=True)

examples = [
    ('Faraday EMF',                 sp.Symbol('B') * sp.Symbol('A') * omega * sp.sin(omega * t)),
    ('Stefan-Boltzmann (T**4)',     sp.Symbol('T') ** 4),
    ('Sigmoid',                     1 / (1 + sp.exp(-x))),
    ('Damped oscillator',           sp.Symbol('A') * sp.exp(-alpha * t) * sp.cos(omega * t)),
    ('Triple Gaussian (cone-like)', sum(sp.exp(-((x - sp.Symbol(f'mu{i}')) / sp.Symbol(f'sig{i}'))**2)
                                         for i in range(1, 4))),
]

print(f"{'Expression':<32} {'pfaffian_r':>11} {'max_path_r':>11} {'eml_depth':>10} {'cost_class':<14}")
print('-' * 80)
for label, expr in examples:
    a = analyze(expr)
    cls = fingerprint_axes(expr)
    print(f'  {label:<30} {a.pfaffian_r:>11} {a.max_path_r:>11} {a.eml_depth:>10}  {cls}')

In [ ]:
# Cell 4 — analyze_dynamics() decomposes an expression into independent
# oscillation / decay modes. Calibrated on E-196 Phase 4f (n=175 elementary
# expressions, Spearman ρ = +0.890); extended in C-207 with the tower-base
# offset to combined ρ = +0.885 across n=193 (elementary + Pfaffian).
from eml_cost import analyze_dynamics

A, alpha_, omega_, t_ = sp.symbols('A alpha omega t', real=True)
expr = A * sp.exp(-alpha_ * t_) * sp.cos(omega_ * t_)

p = analyze_dynamics(expr)
print(f'Expression: {expr}')
print(f'  n_oscillations = {p.n_oscillations}')
print(f'  n_decays       = {p.n_decays}')
print(f'  predicted r    = {p.predicted_r}   (slope-2 rule: 2*n_osc + 1*n_decay)')
print(f'  actual r       = {p.actual_r}')
print(f'  confidence     = {p.confidence}')
print(f'  description    = {p.description}')

In [ ]:
# Cell 5 — integer construction cost from {1}.
# Empirical fit (E-200, n in [1, 10000]):
#   pure-EML:     cost(N) ~ 23.60 * log2(N) - 10.95   (R² = 0.984)
#   BEST routed:  cost(N) ~  4.22 * log2(N) - 1.92    (R² = 0.962)
# Highly composite numbers (HCN) are cheapest; primes are most expensive.
# 360 = 2^3 * 3^2 * 5 — six divisor pairs, cheap to factor.
import math

ADD_BEST, MUL_BEST = 2, 2  # per data/superbest.md (positive-domain v5.2)

def best_cost(n: int, memo: dict[int, int] = None) -> int:
    """Optimal BEST-routed node count to build n from {1}.
    Tries add(a, b) and mul(a, b) for all factorisations and picks the
    minimum. Memoised because branches share sub-results.
    """
    if memo is None:
        memo = {1: 0}
    if n in memo:
        return memo[n]
    best = math.inf
    # Multiplication: n = a*b for divisors
    for a in range(2, int(math.isqrt(n)) + 1):
        if n % a == 0:
            best = min(best, best_cost(a, memo) + best_cost(n // a, memo) + MUL_BEST)
    # Addition: n = a + b, a + b = n, b >= a
    for a in range(1, n // 2 + 1):
        best = min(best, best_cost(a, memo) + best_cost(n - a, memo) + ADD_BEST)
    memo[n] = int(best)
    return memo[n]

n = 360
cost = best_cost(n)
log2_n = math.log2(n)
ratio = cost / log2_n
predicted = 4.22 * log2_n - 1.92

print(f'BEST cost of building {n} from {{1}}:  {cost} nodes')
print(f'  log2({n})           = {log2_n:.3f}')
print(f'  cost / log2(n)      = {ratio:.3f}    (HCN asymptote ~3.90)')
print(f'  predicted (E-200)   = {predicted:.2f}    (4.22 * log2(N) - 1.92)')

print()
print("Comparison: 360 (HCN: 24 divisors) vs 359 (prime):")
print(f'  cost(360) = {best_cost(360):>3} nodes   (highly composite, cheap)')
print(f'  cost(359) = {best_cost(359):>3} nodes   (prime, expensive)')

## What's next

  - **Tower extensions** at `monogate.org/atlas` — 8 Pfaffian generators reach 35 special functions (erf, Bessel, Airy, Gamma, Lambert W, elliptic, hypergeometric).
  - **Dynamics counter** — `analyze_dynamics()` is the public API for the slope-2 rule, validated at Spearman ρ = 0.885 across n=193 expressions in C-207.
  - **eml-cost lint** — `eml-cost lint <file.py>` predicts wall-time of `sympy.simplify` / `factor` / `expand` calls before they run, so pre-commit hooks can warn before a 5-second compile.
  - **Cross-domain siblings** — `find_siblings(expr)` returns the structural neighbours of any expression across the bundled 576-row 12-subdomain corpus.

Honest caveats: every numerical claim above is **OBSERVATION**-tier (empirical fit). The **THEOREM**-tier results are in the 14 Lean files at `monogate-lean/MonogateEML/` and surfaced at `monogate.org/proofs`.